# Phase 0 — EDA & Recall-Ceiling Probe

Full recall-ceiling probe over the **dev** split (incl. dense-text). Loads data from **Hugging Face** with the cache on **Google Drive** (so re-runs are fast) and writes all outputs to Drive. GPU runtime recommended.

Spec: `.claude/documents/features/20_P0_eda_recall_probe.md`.

## 1. Mount Google Drive (data cache + outputs persist here)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE = '/content/drive/MyDrive/recsys2026'
os.environ['HF_HOME'] = f'{DRIVE}/hf_cache'      # HF dataset+model cache on Drive (persists)
OUTPUT_ROOT = f'{DRIVE}/outputs'                 # reports / tables saved here
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
os.makedirs(OUTPUT_ROOT, exist_ok=True)
print('HF cache ->', os.environ['HF_HOME']); print('outputs ->', OUTPUT_ROOT)

## 2. Clone repo@branch + install

In [ ]:
REPO='https://github.com/orrimoch/recsys2026-lora-tutorial.git'; BRANCH='fresh-start'
!git clone --branch $BRANCH --depth 1 $REPO /content/recsys2026 2>/dev/null || (cd /content/recsys2026 && git pull)
%cd /content/recsys2026
!pip -q install datasets bm25s scipy scikit-learn sentence-transformers numpy pandas
import sys; sys.path.insert(0, '.')
# (optional) set HF_TOKEN to avoid rate limits: os.environ['HF_TOKEN']='hf_...'
# --- API keys from Colab Secrets (Settings > Secrets); never hardcode ---
from google.colab import userdata
def _secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None
_hf = _secret('HF_TOKEN')
if _hf:
    os.environ['HF_TOKEN'] = os.environ['HUGGINGFACE_HUB_TOKEN'] = _hf
    from huggingface_hub import login; login(_hf); print('HF authenticated')
else:
    print('HF_TOKEN not in Colab Secrets — public datasets still work but may be rate-limited')
# Gemini (only needed by responder/enrichment notebooks): _secret('GEMINI_API_KEY')


## 3. Config

In [ ]:
DEV_SPLIT='test'; DEPTH=500; KS=[20,50,100,200,500]; COLD_THRESHOLD=1
DENSE_MODEL='BAAI/bge-large-en-v1.5'
# BGE-en-v1.5 is ASYMMETRIC: the QUERY needs this instruction prefix, the DOC gets none.
# Omitting it silently depresses dense recall (R4 §4.3). Empty string for non-BGE encoders.
DENSE_QUERY_PREFIX='Represent this sentence for searching relevant passages: '
CONTENT_MODALITY='metadata-qwen3_embedding_0.6b'
ORG='talkpl-ai'
ENRICHED_GLOB=f'{OUTPUT_ROOT}/catalog_enriched_*.parquet'   # A1 output (newest used); probe the ENRICHED ceiling

## 4. Load data from HF (cached to Drive) → F1 loaders (no load_from_disk)

In [ ]:
import glob, pandas as pd
from datasets import load_dataset
from mcrs.data.catalog import Catalog
from mcrs.data.embeddings import TrackEmbeddings, UserEmbeddings
from mcrs.data.conversations import Conversations

meta_rows = load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Metadata', split='all_tracks')
# Layer A1 enrichment onto the catalog so BM25/dense probe the ENRICHED ceiling (not raw).
_enr_files = sorted(glob.glob(ENRICHED_GLOB))
if _enr_files:
    _edf = pd.read_parquet(_enr_files[-1])
    enr = dict(zip(_edf['track_id'], _edf['enriched_doc']))
    cat = Catalog(meta_rows, enriched_docs=enr)
    USE_ENRICHED = True
    print(f'enriched docs: {len(enr)} (from {_enr_files[-1].split("/")[-1]}) -> probing ENRICHED')
else:
    cat = Catalog(meta_rows)
    USE_ENRICHED = False
    print('WARNING: no enriched parquet found in', ENRICHED_GLOB, '-> probing RAW docs (run A1 first)')

tre = load_dataset(f'{ORG}/TalkPlayData-Challenge-Track-Embeddings', split='all_tracks')
te_content = TrackEmbeddings(tre.select_columns(['track_id', CONTENT_MODALITY]), modalities=[CONTENT_MODALITY])
te_cf = TrackEmbeddings(tre.select_columns(['track_id', 'cf-bpr']), modalities=['cf-bpr'])
ue_dd = load_dataset(f'{ORG}/TalkPlayData-Challenge-User-Embeddings')
ue_rows = [r for sp in ue_dd for r in ue_dd[sp]]
ue = UserEmbeddings(ue_rows)
conv = Conversations(load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset', split=DEV_SPLIT), cold_threshold=COLD_THRESHOLD)
print('catalog', len(cat))

## 5. EDA questions (§5) — sizes the design parameters

In [ ]:
import numpy as np
turns = list(conv.turns())
golds = [conv.gold(t.session_id, t.turn_number) for t in turns]
segs  = [t.segment for t in turns]
print('catalog_size =', len(cat))
print('dev turns =', len(turns), '| cold/warm =', segs.count('cold'), '/', segs.count('warm'))
print('golds_in_catalog =', sum(g in cat for g in golds), '/', len(turns))
gih = sum(1 for t,g in zip(turns,golds) if g in set(t.history_tids))
print('gold_in_history_rate =', round(gih/len(turns),4), '-> L1 history-rule default')
ctx_len = np.array([sum(len(u.split()) for u in t.utterances) for t in turns])
print('ctx tokens p50/p90/p95/p99/max =', [int(np.percentile(ctx_len,p)) for p in (50,90,95,99,100)])
tr_ids = set(load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset', split='train')['session_id'])
print('train∩dev sessions =', len(tr_ids & set(t.session_id for t in turns)), '(expect 0)')

## 6. Build channels (R1/R3/R4/R5 + R6 related_artist) + run probe (R7)

In [ ]:
import os, pickle
from sentence_transformers import SentenceTransformer
from mcrs.retrieval.query import QueryBuilder
from mcrs.retrieval.bm25_channel import BM25Channel
from mcrs.retrieval.dense_channel import DenseChannel
from mcrs.retrieval.personalization import ContentKNNChannel, CFChannel, SameArtistChannel
from mcrs.retrieval.related_artist import RelatedArtistChannel, build_artist_cooc, tid_to_artists_from_catalog

qb = QueryBuilder()
queries = [qb.build(t).text for t in turns]
bc = [{'history_tids': t.history_tids, 'user_id': t.user_id} for t in turns]
uids = [t.user_id for t in turns]

model = SentenceTransformer(DENSE_MODEL, device='cuda')
# doc side: enriched docs, NO prefix (BGE asymmetric). query side: prepend DENSE_QUERY_PREFIX.
doc_mat = model.encode([cat.id_to_metadata(t, enriched=USE_ENRICHED) for t in cat.index_to_id],
                       batch_size=256, normalize_embeddings=True, show_progress_bar=True)
dense = DenseChannel(cat.index_to_id, doc_mat,
                     lambda qs: model.encode([DENSE_QUERY_PREFIX + q for q in qs],
                                             batch_size=256, normalize_embeddings=True), normalize=False)

# R6 related-artist: cross-session artist co-occurrence from TRAIN (train-only -> no leak), cached to Drive.
COOC_PKL = f'{OUTPUT_ROOT}/artist_cooc.pkl'
if os.path.exists(COOC_PKL):
    with open(COOC_PKL, 'rb') as f: cooc = pickle.load(f)
    print('loaded artist co-occurrence:', len(cooc), 'artists')
else:
    _train = load_dataset(f'{ORG}/TalkPlayData-Challenge-Dataset', split='train')
    cooc = build_artist_cooc(_train, tid_to_artists_from_catalog(cat))
    with open(COOC_PKL, 'wb') as f: pickle.dump(cooc, f)
    print('built + cached artist co-occurrence:', len(cooc), 'artists ->', COOC_PKL)
related = RelatedArtistChannel(cat, cooc)

In [ ]:
per = {
  'bm25':         BM25Channel(cat, enriched=USE_ENRICHED).batch_text_to_item_retrieval(queries, DEPTH),
  'dense':        dense.batch_text_to_item_retrieval(queries, DEPTH),
  'content_knn':  ContentKNNChannel(te_content, CONTENT_MODALITY).batch_text_to_item_retrieval(queries, DEPTH, batch_context=bc),
  'cf':           CFChannel(ue, te_cf, 'cf-bpr').batch_text_to_item_retrieval(queries, DEPTH, user_ids=uids),
  'same_artist':  SameArtistChannel(cat).batch_text_to_item_retrieval(queries, DEPTH, batch_context=bc),
  'related_artist': related.batch_text_to_item_retrieval(queries, DEPTH, batch_context=bc),
}

## 7. Recall-ceiling table + DESIGN PARAMETERS → save to Drive

In [ ]:
import pandas as pd
from mcrs.eval.probe import recall_ceiling
rep = recall_ceiling(per, golds, ks=KS, segments=segs)

# --- overall per-channel + fused recall ---
rows=[{'channel':l, **{f'r@{k}':round(e['recall'][k],3) for k in KS}, 'unique':round(e['unique_recall'],3)} for l,e in rep['per_channel'].items()]
rows.append({'channel':'FUSED', **{f'r@{k}':round(rep['fused']['recall'][k],3) for k in KS}})
tbl = pd.DataFrame(rows); print('=== OVERALL recall-ceiling ==='); print(tbl.to_string(index=False))
fused = rep['fused']['recall']
fusion_k = next((k for k in KS if fused[k] >= 0.90), None)
print('\nfused r@200=', round(fused[200],3), '| r@500=', round(fused[500],3), '| fusion_k(>=0.90):', fusion_k)

# --- BY-SEGMENT recall (cold vs warm): the §7.3/§10 per-segment ablation ---
segs_present = sorted(set(segs))                       # e.g. ['cold','warm']
SEG_KS = [k for k in (100, 200) if k in KS] or [KS[-1]]
def _seg_row(label, by_seg):
    row = {'channel': label}
    for s in segs_present:
        for k in SEG_KS:
            row[f'{s} r@{k}'] = round(by_seg.get(s, {}).get(k, float('nan')), 3)
    return row
seg_rows = [_seg_row(l, e.get('by_segment', {})) for l, e in rep['per_channel'].items()]
seg_rows.append(_seg_row('FUSED', rep['fused'].get('by_segment', {})))
seg_tbl = pd.DataFrame(seg_rows)
print(f"\n=== BY-SEGMENT recall (cold/warm = {segs.count('cold')}/{segs.count('warm')}) — drives segment-aware fusion weights (plan §7.3 #3 / §10) ===")
print(seg_tbl.to_string(index=False))
print("READ: a channel strong on warm but ~0 on cold (history channels: cf/content_knn/same_artist/related_artist) -> up-weight on warm, ~0 on cold; query channels (bm25/dense) carry cold.")

# --- save ---
with open(f'{OUTPUT_ROOT}/eda.md','w') as f:
    f.write('# P0 Recall-Ceiling (full)\n\n## Overall\n\n'+tbl.to_markdown(index=False))
    f.write('\n\n## By segment (cold vs warm)\n\n'+seg_tbl.to_markdown(index=False)+'\n\n## DESIGN PARAMETERS\n')
    f.write(f'- fusion_k = {fusion_k} (None => 0.90 not reached; route to A1/R6)\n')
    f.write(f'- topk_internal = {DEPTH} (>= fusion_k)\n- cold_threshold = {COLD_THRESHOLD}; cold/warm = {segs.count("cold")}/{segs.count("warm")}\n')
    f.write(f'- gold_in_history_rate = {round(gih/len(turns),4)}\n- channel_keep_list: drop ~0-unique channels above\n')
    f.write('- segment_weights: set per-channel cold/warm weights iff the by-segment table shows a per-segment win (plan §7.3 #3)\n')
tbl.to_csv(f'{OUTPUT_ROOT}/recall_ceiling.csv', index=False)
seg_tbl.to_csv(f'{OUTPUT_ROOT}/recall_ceiling_by_segment.csv', index=False)
print('\nsaved ->', OUTPUT_ROOT, '(eda.md, recall_ceiling.csv, recall_ceiling_by_segment.csv)')

## 8. Gate
**recall@200 ≥ 0.90** (and r@20 ≥ 0.75) → retrieval gate met, proceed to rerank. Else the recall wall binds → prioritize A1 enrichment/doc2query + R6 extension channels, re-probe.